## Colab session configuration

Connect to google cloud runtime and use GPU.

## Reading hg38 genome assembly 

Set parent directory.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import pathlib
from pathlib import Path
!pip install pyfaidx
from pyfaidx import Fasta

# Function to get the root directory
def get_root():
    """
    Get the root directory to access all files using Colab or local.

    Note that this function assumes the dir containing notebook is 1 below repo main dir.
    """
    COLAB = os.path.abspath(".") == "/content"

    if COLAB:
        root = "/content"
    else:
        root = Path.cwd().parent

    return root

# Set root
root = get_root()

# Set 
if root not in sys.path:
    sys.path.insert(0, str(root))

Download repo and the hg38 fasta zipped file.

Figure out a good way to run this pipeline only if colab is detected, then otherwise, just use local data folder.

In [ ]:
# Download repo to access scripts
!git clone https://github.com/eddykang06/mini-gLM.git
&cd mini-gLM

# Add repo scripts to path
sys.path.append("/content/mini-gLM")

# Download zipped human genome files
!mkdir -p data
!curl -L -C - "https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz" -o data/hg38.fa.gz
!curl -L -C - "https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes" -o data/hg38.chrom.sizes
!gunzip -c data/hg38.fa.gz > data/hg38.fa # Unzip to fasta

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  938M  100  938M    0     0  63.9M      0  0:00:14  0:00:14 --:--:-- 60.9M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 11672  100 11672    0     0  93291      0 --:--:-- --:--:-- --:--:-- 94129


Sample random sections of the genome, weighted by chromosome length and within the specified sequence length range.

In [ ]:
from  import sample_from_fasta

# Set seed
rng = np.random.default_rng(111)

# Data directory
data_dir = root / "data"
fasta_file = data_dir / "hg38.fa"
chrom_size_file = data_dir / "hg38.chrom.sizes"

# Set parameters
n_seqs = 100
min_length = 20
max_length = 500

# Get pretraining data
pretraining = sample_from_fasta(
    fasta_file = fasta_file,
    chrom_size_file = chrom_size_file,
    n_seqs = n_seqs,
    min_length = min_length,
    max_length = max_length
)

ModuleNotFoundError: No module named 'src'

## Tokenizer

Wrap BPE into a function.

In [ ]:
from src.tokenize import (
    find_all_adjacent_pairs,
    merge_token_pairs
)

def bpe_tokenize_dna(
        sequence_list: list[str], 
        final_vocab_size: int, 
        seed : int
) -> tuple[list[str], dict[tuple, str]]:
    
    rng = np.random_default_rng(seed)
    merge_rules = {}

    # Pre-tokenization of sequences is done with simple initial vocabulary
    vocab = ["A", "C", "G", "T"]
    split_corpus = [list(seq) for seq in sequence_list]

    # Iterate until vocab reaches desired size
    while len(vocab) < final_vocab_size:
        
        # Store all pair counts in corpus
        pair_counts_all = {}

        for seq in split_corpus:

            # Find counts of all adjacent pairs and add to overall pair counts
            pair_counts_seq = find_all_adjacent_pairs(seq)

            for pair, count in pair_counts_seq.items():
                pair_counts_all[pair] = pair_counts_all.get(pair, 0) + count

    # Store pairs and counts
    pairs = list(pair_counts_all.keys())
    counts = np.array(list(pair_counts_all.values()))

    # Find pair(s) with the highest counts
    best_idx = np.argwhere(counts == counts.max()).ravel()

    # If multiple pairs are found, then randomly select one
    if len(best_idx) > 1:
        random_idx = rng.choice(best_idx)
        best_pair = pairs[random_idx]
    
    else:
        best_idx = best_idx[0]
        best_pair = pairs[best_idx]
    
    # Join all instances of the pair in split_corpus
    split_corpus = [merge_token_pairs(seq, best_pair) for seq in split_corpus]

    # Add new token to vocbaulary if its not redundant
    new_token = "".join(best_pair)
    vocab.append(new_token)

    # Store merge rule
    merge_rule = {best_pair: new_token}
    merge_rules.update(merge_rule)

    return vocab, merge_rules

In [ ]:
corpus = [
    "ACGTACTACATACTAA",
    "ACGTACAGACGTAACCC",
    "AATTG",
    "AGGCATACC",
    "TTGGTTAAAAGGAA"
]

vocab, rules = bpe_tokenize_dna(
    sequence_list = corpus,
    
)

In [ ]:
from src.tokenize import (
    find_all_adjacent_pairs,
    merge_token_pairs
)

# Set seed for random selection
seed = 111
rng = np.random.default_rng(seed)

# Function to implement BPE algorithm
# Sample corpus
corpus = [
    "ACGTACTACATACTAA",
    "ACGTACAGACGTAACCC",
    "AATTG",
    "AGGCATACC",
    "TTGGTTAAAAGGAA"
]

# Initial vocab
vocab = ["A", "C", "G", "T"]

# Set size of interest
final_size = 10

# Store merge rules for future tokenization
merge_rules = {}

# In iteration 1, simple split each sequence into list (treat each sequence as a word)
split_corpus = [list(seq) for seq in corpus]

# Iterate until vocab reaches the specified length
while len(vocab) < final_size:

    # Dictionary to store the counts for each pair seen in the corpus
    pair_counts_all = {}

    # For each sequence in the corpus, find the unique pairs and store them a lists
    for seq in split_corpus:

        # Find counts of all adjacent pairs in sequence and store as a dictionary
        pair_counts_seq = find_all_adjacent_pairs(seq)

        for pair, count in pair_counts_seq.items():
            
            # Add the key as a new pair if not yet seen, otherwise simply add to the counts
            pair_counts_all[pair] = pair_counts_all.get(pair, 0) + count
 
    # Store pairs and counts
    pairs = list(pair_counts_all.keys())
    counts = np.array(list(pair_counts_all.values()))

    # Find pair(s) with the highest counts
    best_idx = np.argwhere(counts == counts.max()).ravel()

    # If multiple pairs are found, then randomly select one
    if len(best_idx) > 1:
        random_idx = rng.choice(best_idx)
        best_pair = pairs[random_idx]
    
    else:
        best_idx = best_idx[0]
        best_pair = pairs[best_idx]
    
    # Join all instances of the pair in split_corpus
    split_corpus = [merge_token_pairs(seq, best_pair) for seq in split_corpus]

    # Add new token to vocbaulary if its not redundant
    new_token = "".join(best_pair)
    vocab.append(new_token)

    # Store merge rule
    merge_rule = {best_pair: new_token}
    merge_rules.update(merge_rule)

print(vocab)
print(merge_rules)

['A', 'C', 'G', 'T', 'AC', 'AA', 'TAC', 'TT', 'ACG', 'GG']
{('A', 'C'): 'AC', ('A', 'A'): 'AA', ('T', 'AC'): 'TAC', ('T', 'T'): 'TT', ('AC', 'G'): 'ACG', ('G', 'G'): 'GG'}


In [ ]:
def seqs_to_corpus():

    
    return corpus



class BPE_Tokenizer():
    def __init__(self, vocab, vocab_size):
        self.vocab = self.vocab
        self.vocab_size = len(self.vocab)
        self.merge_rules = self.merge_rules
    
    # Fit tokenizer on list of sequences
    def fit(self, sequence_list, desired_size, seed):

    
    # Tokenize a given sequence
    def tokenize(self, seqs):




In [ ]:
class object:
    def __init__(self, stat):
        self.stat = stat



Reference: https://huggingface.co/learn/llm-course/en/chapter6/5

## Model

In [ ]:
!pip install accelerate
!pip install datasets
!pip install transformers
!pip install torch
!pip install flash-attn

import accelerate
import flash_attn
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

torch.backends.cudnn.benchmark=True

Implement a sparse attention transformer.

Implement the model with 10 transformer blocks.

In [ ]:
class miniglm(nn.Module):

    def __init__(self):